# Controlled Descent Simulator — the control-limited iLQR solver

*Vehicle-agnostic notebook.* This notebook develops the nonlinear
**Model-Predictive Control (MPC)** solver used across the simulator — a
**control-limited iLQR / DDP** algorithm — **independently of any specific
vehicle**, validates it on a synthetic benchmark, and then **conforms it against
the hand-written C++** in `libs/control/ilqr.hpp`.

**MPC** (Model-Predictive Control): at each instant we look a fixed number of
steps into the future, solve an optimal-control problem over that *horizon*,
apply only the first command, then repeat one step later (*receding horizon*).
**iLQR** (iterative Linear-Quadratic Regulator) and **DDP** (Differential
Dynamic Programming) are the algorithms we use to solve that problem when the
dynamics are nonlinear and the actuators are bounded.

**Why this notebook exists (methodology).** The vehicle *model* is a symbolic
object, so it is **generated** (SymPy → C++ codegen, see the `model/`
notebooks). The *solver* is an **algorithm**, not a formula — there is nothing
to "generate" — so it is **hand-written in C++** as reusable infrastructure and
**validated from Python**. This notebook is the Python side: the pedagogical
derivation (the "spec") and the conformance oracle. The C++ is the product; the
two are pinned together by the conformance test at the end.

> Reading order is linear: every term is defined where it first appears, and no
> quantity is used before it is motivated.

References: Jacobson & Mayne, *Differential Dynamic Programming* (1970);
Li & Todorov, *Iterative LQG* (ICINCO 2004); Tassa, Erez & Todorov,
*Control-limited DDP* (ICRA 2014); Rawlings, Mayne & Diehl, *Model Predictive
Control* (2nd ed., 2017).

## 1. The problem: finite-horizon optimal control with input limits

We work in **discrete time**. Let $x_k \in \mathbb{R}^{n_x}$ be the state and
$u_k \in \mathbb{R}^{n_u}$ the command at step $k$. One step of the plant is a
map

$$ x_{k+1} = F(x_k, u_k), $$

which we obtain by integrating the continuous dynamics $\dot x = f(x,u)$ over a
fixed step $\Delta t$ (here with classic RK4). Over a horizon of $N$ steps we
choose the command sequence $u_0,\dots,u_{N-1}$ to minimise

$$ J = \sum_{k=0}^{N-1} \ell(x_k, u_k) \;+\; \ell_f(x_N), $$

subject to the dynamics and to **box constraints** on the actuators
$u^{\min} \le u_k \le u^{\max}$ (motors saturate; this is not optional). Here
$\ell$ is the **stage cost** (how much we dislike being off-target and using
effort at step $k$) and $\ell_f$ the **terminal cost** (a stand-in for "all the
cost after the horizon ends" — a finite horizon needs one, or the controller is
short-sighted).

In MPC we solve this from the *current* state $x_0$, apply $u_0$, advance one
step, and re-solve — so the solver runs once per control tick and must be fast.

## 2. From LQR to iLQR

If $F$ were linear and $\ell,\ell_f$ quadratic, the exact minimiser would be the
classic **LQR** (Linear-Quadratic Regulator): a backward Riccati recursion gives
an affine feedback law $u_k = -K_k x_k$ in one shot. Our $F$ is nonlinear, so we
**iterate LQR**: around a current guess trajectory we build the *local*
linear-quadratic approximation, solve that exactly for an improving step, roll
it forward through the *true* nonlinear dynamics, and repeat. That is **iLQR**;
adding the second-order dynamics terms would make it **DDP** — we use the iLQR
(Gauss-Newton) variant, which needs only first derivatives of $F$.

One iteration has two sweeps:

- **Backward pass** — from $k=N$ down to $0$, propagate a local quadratic model
  of the *cost-to-go* (the **value function** $V_k$: the best achievable cost
  from step $k$ onward) and read off, at each knot, a feedforward step
  $\mathbf{k}_k$ and a feedback gain $K_k$.
- **Forward pass** — from $k=0$ up, apply $u_k \leftarrow u_k + \alpha\,
  \mathbf{k}_k + K_k (x_k - \bar x_k)$ through the true dynamics, with a
  step size $\alpha$ chosen by line search.

<svg xmlns="http://www.w3.org/2000/svg" width="560" height="150" viewBox="0 0 560 150">
<rect width="560" height="150" fill="#ffffff"/>
<text x="10" y="22" font-family="sans-serif" font-size="14" fill="#111">One iLQR iteration</text>
<!-- knots -->
<g font-family="sans-serif" font-size="12" fill="#111">
<circle cx="70" cy="80" r="6" fill="#1f77b4"/><text x="58" y="110">x0</text>
<circle cx="190" cy="80" r="6" fill="#1f77b4"/><text x="182" y="110">xk</text>
<circle cx="310" cy="80" r="6" fill="#1f77b4"/><text x="300" y="110">xk+1</text>
<circle cx="470" cy="80" r="6" fill="#1f77b4"/><text x="462" y="110">xN</text>
</g>
<!-- backward arrow -->
<line x1="470" y1="55" x2="70" y2="55" stroke="#d62728" stroke-width="2" marker-end="url(#a)"/>
<text x="210" y="46" font-family="sans-serif" font-size="12" fill="#d62728">backward pass: value function Vk, gains (kk, Kk)</text>
<!-- forward arrow -->
<line x1="70" y1="105" x2="470" y2="105" stroke="#2ca02c" stroke-width="2" marker-end="url(#a)"/>
<text x="150" y="128" font-family="sans-serif" font-size="12" fill="#2ca02c">forward pass: roll true dynamics, line-search alpha</text>
<defs><marker id="a" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
<path d="M0,0 L6,3 L0,6 Z" fill="#555"/></marker></defs>
</svg>

*The backward pass is the LQR of the local quadratic model; the forward pass is
where the nonlinearity re-enters.*

### 2.1 The backward pass, one knot at a time

At knot $k$ we have the discrete Jacobians $A_k = \partial F/\partial x$ and
$B_k = \partial F/\partial u$ (built below), the stage-cost derivatives
($\ell_x \equiv \partial \ell/\partial x$, $\ell_{xx} \equiv \partial^2
\ell/\partial x^2$, and likewise $\ell_u, \ell_{uu}$), and the quadratic value
model at the *next* knot, $(V_x, V_{xx})$. We form the local **action-value**
model $Q$ (cost of acting now plus the value of where we land):

$$
\begin{aligned}
Q_x &= \ell_x + A_k^\top V_x, & Q_u &= \ell_u + B_k^\top V_x,\\
Q_{xx} &= \ell_{xx} + A_k^\top V_{xx} A_k, & Q_{uu} &= \ell_{uu} + B_k^\top V_{xx} B_k,\\
& & Q_{ux} &= B_k^\top V_{xx} A_k.
\end{aligned}
$$

The unconstrained minimiser over the control step is
$\delta u = -Q_{uu}^{-1}(Q_u + Q_{ux}\,\delta x)$, i.e. feedforward
$\mathbf{k}_k = -Q_{uu}^{-1} Q_u$ and feedback $K_k = -Q_{uu}^{-1} Q_{ux}$. With
input limits we instead solve a **box-QP** for $\mathbf{k}_k$ (§3), and put the
feedback only on the components left free. The value model is then pushed back
one step. We keep $\ell_{xx},\ell_{uu}$ positive-semidefinite by writing the
cost in **Gauss-Newton** (squared-residual) form, so $Q_{uu}$ is invertible.

## 3. Actuator limits: the box-QP (Tassa, Erez & Todorov 2014)

Ignoring limits and then clipping the result is wrong: it discards the coupling
between clipped and free actuators. Instead, at each knot we solve the small
**box-constrained quadratic program**

$$ \min_{\delta u}\; \tfrac12 \delta u^\top Q_{uu}\, \delta u + Q_u^\top \delta u
   \quad\text{s.t.}\quad u^{\min}-u_k \le \delta u \le u^{\max}-u_k, $$

with a **projected-Newton** method: guess which bounds are active, take a Newton
step on the *free* variables (a small Cholesky solve), project back into the
box, and repeat until the active set settles. The feedback gain $K_k$ is applied
only to the free components — a clamped motor does not react to state error.

## 4. Robustness: line search, Levenberg regularisation, warm start

Three ingredients turn the raw recursion into something that reliably descends:

- **Line search.** The backward step is only a local model, so in the forward
  pass we try $\alpha \in \{1, \tfrac12, \tfrac14, \dots\}$ and accept the first
  that lowers the true cost $J$.
- **Levenberg regularisation.** If $Q_{uu}$ is nearly singular we add
  $\mu I$ to it (a trust-region-like damping): raise $\mu$ when a step fails,
  lower it when steps succeed.
- **Warm start.** In receding-horizon use the previous solve is an excellent
  guess: we shift it by one step and start the next solve from there — a few
  iterations then suffice. The warm-start buffer is owned by the *caller*.

## 5. A vehicle-agnostic Python reference

The reference implementation is packaged in **`ilqr_ref.py`** (next to this
notebook) — the *single source* of the Python solver, mirroring the C++ in
`libs/control/ilqr.hpp` one-for-one. It knows nothing about quadrotors: it takes
the plant `f`, its Jacobians `jac`, an optional state `project`ion (e.g.
renormalising a quaternion — a no-op here), the stage/terminal costs, the box,
and the warm start. Packaging it as a module (rather than re-typing it here)
lets the *same* code be reused by the `model/` notebooks for their closed-loop
demos, and keeps this notebook focused on the *why*.

Its public surface — all mirrored in the C++:

- `rk4_step(x, u, f, dt)` — one integration step;
- `sensitivity(x, u, f, jac, dt)` → `A, B` — discrete Jacobians (§2.1);
- `box_qp(H, g, lo, hi)` → `z, free` — the actuator box-QP (§3);
- `backward_pass(...)`, `forward_pass(...)` — the two sweeps (§2, §4);
- `ilqr(...)` → `xs, us, hist` — a full solve from a warm start;
- `ilqr_solve(...)` → `u0, xs, us, J` — the receding-horizon entry point.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))     # ilqr_ref.py sits next to this notebook
import numpy as np
from ilqr_ref import (rk4_step, sensitivity, fd_sensitivity, box_qp,
                      backward_pass, forward_pass, ilqr, ilqr_solve)
print('imported the reference solver from ilqr_ref.py')

### 5.1 Acid test — the sensitivity against finite differences (many points)

Before trusting $A_k, B_k$ we check the RK4 chain-rule sensitivity
(`ilqr_ref.sensitivity`) against a central finite difference of the RK4 step
(`ilqr_ref.fd_sensitivity`), at several random points. We report the worst
error (run in §6, once the benchmark `f`/`jac` exist).

## 6. A synthetic benchmark (meaningless by design)

To exercise the *solver* — not to model anything — we use three damped
oscillators, coupled in a ring, with a cubic softening term so the dynamics are
genuinely nonlinear (iLQR must iterate) and the Jacobian is state-dependent.
State $x=[p_0,v_0,p_1,v_1,p_2,v_2]$, one bounded actuator per channel. The exact
same constants are compiled into `libs/control/bind/ilqr_bench.cpp`, so §7 can
compare like with like.

In [ ]:
OMEGA = np.array([1.3, 0.8, 1.7]); ZETA = np.array([0.10, 0.05, 0.15])
GAIN  = np.array([1.0, 1.2, 0.9]);   BETA = np.array([0.20, 0.35, 0.15])
KAPPA = 0.4
QP, QV, RU, WTERM = 3.0, 0.5, 0.08, 15.0
DT, N = 0.05, 25

def bench_f(x, u):
    d = np.zeros(6)
    for i in range(3):
        p, v, pn = x[2*i], x[2*i+1], x[2*((i+1) % 3)]
        d[2*i]   = v
        d[2*i+1] = (-OMEGA[i]**2*p - 2*ZETA[i]*OMEGA[i]*v
                    + GAIN[i]*u[i] - BETA[i]*p**3 + KAPPA*(pn - p))
    return d

def bench_jac(x, u):
    fx = np.zeros((6, 6)); fu = np.zeros((6, 3))
    for i in range(3):
        pi, vi, ni = 2*i, 2*i+1, 2*((i+1) % 3)
        fx[pi, vi] = 1.0
        fx[vi, pi] = -OMEGA[i]**2 - 3*BETA[i]*x[pi]**2 - KAPPA
        fx[vi, vi] = -2*ZETA[i]*OMEGA[i]
        fx[vi, ni] += KAPPA
        fu[vi, i]  = GAIN[i]
    return fx, fu

def bench_stage(x, u, k):
    Wx = np.array([QP, QV]*3)
    lx = Wx*x; lxx = np.diag(Wx)
    lu = RU*u; luu = RU*np.eye(3)
    val = 0.5*(x @ (Wx*x) + RU*(u @ u))
    return val, lx, lxx, lu, luu

def bench_term(x):
    Wx = WTERM*np.array([QP, QV]*3)
    return 0.5*(x @ (Wx*x)), Wx*x, np.diag(Wx)

# now the deferred §5.1 acid test, on the benchmark
worst = 0.0
rng = np.random.default_rng(0)
for _ in range(12):
    xr = rng.normal(size=6); ur = rng.normal(size=3)
    A1, B1 = sensitivity(xr, ur, bench_f, bench_jac, DT)
    A2, B2 = fd_sensitivity(xr, ur, bench_f, DT)
    worst = max(worst, np.abs(A1-A2).max(), np.abs(B1-B2).max())
print(f"sensitivity acid test (12 random points): worst |analytic - FD| = {worst:.2e}")

### 6.1 Run the reference on the benchmark

We regulate to the origin from an offset, once with a loose box (solution
interior) and once with a tight box (the limits bite). We check the box is
respected and plot the commands and the position error.

In [ ]:
import matplotlib.pyplot as plt

def closed_loop(umax, x0, steps=90, max_iters=30):
    lo, hi = -umax*np.ones(3), umax*np.ones(3)
    warm = np.zeros((N, 3))
    x = x0.copy(); U, P = [], []
    for _ in range(steps):
        u0, *_ = ilqr_solve(x, bench_f, bench_jac, lambda s: s,
                            bench_stage, bench_term, lo, hi, DT, max_iters, warm)
        U.append(u0.copy())
        x = rk4_step(x, u0, bench_f, DT)
        P.append([x[0], x[2], x[4]])
    return np.array(U), np.array(P)

x0 = np.array([0.9, 0.0, -1.1, 0.0, 0.7, 0.0])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for umax, style in [(5.0, '-'), (0.6, '--')]:
    U, P = closed_loop(umax, x0)
    box_ok = (np.abs(U) <= umax + 1e-9).all()
    err = np.linalg.norm(P, axis=1)
    ax[0].plot(U[:, 0], style, label=f'u0, umax={umax} (box {"ok" if box_ok else "VIOLATED"})')
    ax[1].plot(err, style, label=f'||pos err||, umax={umax}')
    print(f"umax={umax}: box respected = {box_ok}, final |pos err| = {err[-1]:.4f}")
ax[0].set_title('first command'); ax[0].set_xlabel('tick'); ax[0].legend(fontsize=8)
ax[1].set_title('position error'); ax[1].set_xlabel('tick'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. Conformance against the C++ solver

This is the only C++-aware section. We build the C-ABI shim
(`libs/control/bind/ilqr_bench.cpp`, which instantiates the C++ solver on the
*same* benchmark), load it with `ctypes`, and compare the **converged solution**
of the Python reference against the C++ one.

A caution on what to compare: the solver has **data-dependent branches**
(line-search acceptance, the Levenberg schedule, the active set). A floating
point difference of $10^{-13}$ between NumPy and C++ can flip a branch and make
the *iterate-by-iterate* paths diverge while both remain correct. So we compare
the **converged** first command, its trajectory and cost — not the intermediate
iterates.

In [ ]:
import subprocess, os, ctypes, pathlib

root = pathlib.Path.cwd()
while not (root / 'libs' / 'control').exists() and root != root.parent:
    root = root.parent
bind = root / 'libs' / 'control' / 'bind'
build = root / 'build-ilqr-bind'
subprocess.run(['cmake', '-S', str(bind), '-B', str(build),
                '-DCMAKE_BUILD_TYPE=Release'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
subprocess.run(['cmake', '--build', str(build)], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
lib_path = next(p for p in (build/'libilqr_bench.dylib', build/'libilqr_bench.so') if p.exists())
lib = ctypes.CDLL(str(lib_path))
lib.ilqr_bench_solve.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_double, ctypes.c_int,
                                 ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double),
                                 ctypes.POINTER(ctypes.c_double)]

def cpp_solve(x0, umax, max_iters):
    cx0 = (ctypes.c_double*6)(*x0)
    warm_in = (ctypes.c_double*(N*3))(*([0.0]*(N*3)))
    u0 = (ctypes.c_double*3)(); wout = (ctypes.c_double*(N*3))()
    lib.ilqr_bench_solve(cx0, ctypes.c_double(umax), ctypes.c_int(max_iters), warm_in, u0, wout)
    return np.array([u0[a] for a in range(3)])
print('C++ shim built and loaded:', lib_path.name)

In [ ]:
# same single solve (zero warm start) on both sides; compare the first command
x0 = np.array([0.9, 0.0, -1.1, 0.0, 0.7, 0.0]); MAXIT = 80
print(f"{'scenario':22s}  {'||u0_py - u0_cpp||':>20s}  verdict")
all_ok = True
for label, umax in [('loose box (umax=5.0)', 5.0), ('tight box (umax=0.6)', 0.6)]:
    warm = np.zeros((N, 3))
    u0_py, *_ = ilqr_solve(x0, bench_f, bench_jac, lambda s: s, bench_stage, bench_term,
                           -umax*np.ones(3), umax*np.ones(3), DT, MAXIT, warm)
    u0_cpp = cpp_solve(x0, umax, MAXIT)
    d = np.linalg.norm(u0_py - u0_cpp)
    ok = d < 1e-6; all_ok &= ok
    print(f"{label:22s}  {d:>20.2e}  {'MATCH' if ok else 'MISMATCH'}")
print('\nCONFORMANCE:', 'PASSED' if all_ok else 'FAILED')

## 8. What this gives us

We now have a **generic, vehicle-agnostic iLQR solver** whose Python reference
(this notebook, the spec) and C++ implementation (`libs/control/ilqr.hpp`, the
product) are **pinned together** by the conformance test — future edits to
either side break the test if they drift. The C++ solver is reused by the
runtime models (e.g. the quadrotor MPC) by supplying a *model-specific* cost;
that cost, and the closed-loop demos on the real vehicle, live in the `model/`
notebooks — not here.

A dependency-free version of this certificate
(`libs/control/bind/ilqr_conformance.py`, ctypes + stdlib only) runs the C++
side without a Python solver at all, checking the returned command sequence is a
constrained optimum (a box-projected KKT residual $\approx 0$); it is the CI-able
gate required for every hand-written controller.